<a href="https://colab.research.google.com/github/mx-oscar-hdez/Proyectos/blob/main/Challenge_Asistente_de_Pol%C3%ADticas_con_RAG_Mx_Oscar_Hdez_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets

!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.1 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [6]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [9]:
# Prompt de clasificación en modo few-shot

prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Negativo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Mixto

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento: Positivo"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)
# El formato few-shot suele acotar mejor la salida a una sola palabra de la categoría esperada

Few-shot: **Reseñas clasificadas correctamente**

1. **Reseña:** "Me encantó, llegó rápido y en perfecto estado."  
   **Sentimiento:** Positivo

2. **Reseña:** "Nunca llegó mi pedido, pésimo servicio."  
   **Sentimiento:** Negativo

3. **Reseña:** "El envío llegó tarde pero el producto es excelente."  
   **Sentimiento:** Mixto (tiene aspectos negativos y positivos)


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [10]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso 1 – Entender la situación**  
- Primer tren: sale de la ciudad **A** a \(80 \ \text{km/h}\).  
- Segundo tren: sale 2 h después del primero, también desde **A**, y va a \(120 \ \text{km/h}\).  
- Ambos trenes viajan en la misma dirección hacia el mismo destino.

**Paso 2 – Calcular el desfase inicial**  
El segundo tren parte 2 h después, por lo que en ese momento el primero ya ha avanzado.  
Distancia que el primero ha recorrido en esas 2 h:  
\[
d_{\text{desfase}} = v_{\text{primero}} \times t = 80 \ \text{km/h} \times 2 \ \text{h}= 160 \ \text{km}
\]

**Paso 3 – Relacionar el problema con una distancia a recorrer**  
Una vez que el segundo tren ha salido, la distancia que debe cerrar es exactamente esa brecha de \(160\) km.

**Paso 4 – Encontrar la velocidad relativa**  
La velocidad de acercamiento entre los dos trenes es la diferencia de sus velocidades (porque viajan en la misma dirección):  
\[
v_{\text{relativa}} = 120 \ \text{km/h} - 80 \ \text{km/h} = 40 \ \text{km/h}


In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso 1 – Entender la situación**  
- Primer tren: sale de la ciudad **A** a \(80 \ \text{km/h}\).  
- Segundo tren: sale 2 h después del primero, también desde **A**, y va a \(120 \ \text{km/h}\).  
- Ambos trenes viajan en la misma dirección hacia el mismo destino.

**Paso 2 – Calcular el desfase inicial**  
El segundo tren parte 2 h después, por lo que en ese momento el primero ya ha avanzado.  
Distancia que el primero ha recorrido en esas 2 h:  
\[
d_{\text{desfase}} = v_{\text{primero}} \times t = 80 \ \text{km/h} \times 2 \ \text{h}= 160 \ \text{km}
\]

**Paso 3 – Relacionar el problema con una distancia a recorrer**  
Una vez que el segundo tren ha salido, la distancia que debe cerrar es exactamente esa brecha de \(160\) km.

**Paso 4 – Encontrar la velocidad relativa**  
La velocidad de acercamiento entre los dos trenes es la diferencia de sus velocidades (porque viajan en la misma dirección):  
\[
v_{\text{relativa}} = 120 \ \text{km/h} - 80 \ \text{km/h} = 40 \ \text{km/h}


In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso 1 – Entender la situación**  
- Primer tren: sale de la ciudad **A** a \(80 \ \text{km/h}\).  
- Segundo tren: sale 2 h después del primero, también desde **A**, y va a \(120 \ \text{km/h}\).  
- Ambos trenes viajan en la misma dirección hacia el mismo destino.

**Paso 2 – Calcular el desfase inicial**  
El segundo tren parte 2 h después, por lo que en ese momento el primero ya ha avanzado.  
Distancia que el primero ha recorrido en esas 2 h:  
\[
d_{\text{desfase}} = v_{\text{primero}} \times t = 80 \ \text{km/h} \times 2 \ \text{h}= 160 \ \text{km}
\]

**Paso 3 – Relacionar el problema con una distancia a recorrer**  
Una vez que el segundo tren ha salido, la distancia que debe cerrar es exactamente esa brecha de \(160\) km.

**Paso 4 – Encontrar la velocidad relativa**  
La velocidad de acercamiento entre los dos trenes es la diferencia de sus velocidades (porque viajan en la misma dirección):  
\[
v_{\text{relativa}} = 120 \ \text{km/h} - 80 \ \text{km/h} = 40 \ \text{km/h}


In [ ]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso 1 – Entender la situación**  
- Primer tren: sale de la ciudad **A** a \(80 \ \text{km/h}\).  
- Segundo tren: sale 2 h después del primero, también desde **A**, y va a \(120 \ \text{km/h}\).  
- Ambos trenes viajan en la misma dirección hacia el mismo destino.

**Paso 2 – Calcular el desfase inicial**  
El segundo tren parte 2 h después, por lo que en ese momento el primero ya ha avanzado.  
Distancia que el primero ha recorrido en esas 2 h:  
\[
d_{\text{desfase}} = v_{\text{primero}} \times t = 80 \ \text{km/h} \times 2 \ \text{h}= 160 \ \text{km}
\]

**Paso 3 – Relacionar el problema con una distancia a recorrer**  
Una vez que el segundo tren ha salido, la distancia que debe cerrar es exactamente esa brecha de \(160\) km.

**Paso 4 – Encontrar la velocidad relativa**  
La velocidad de acercamiento entre los dos trenes es la diferencia de sus velocidades (porque viajan en la misma dirección):  
\[
v_{\text{relativa}} = 120 \ \text{km/h} - 80 \ \text{km/h} = 40 \ \text{km/h}


1


In [11]:
# Razonamiento paso a paso (chain-of-thought)

problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca

**Paso a paso:**

1. **Define las variables**  
   - Velocidad del tren 1 (T1): \(v_1 = 80 \text{ km/h}\)  
   - Velocidad del tren 2 (T2): \(v_2 = 120 \text{ km/h}\)  
   - Tiempo que T1 recorre antes de que T2 salga: \(t_{\text{h}1} = 2 \text{ h}\)

2. **Calcular la distancia que T1 recorre en esas 2 horas**  
   \[
   d_{\text{h}1} = v_1 \times t_{\text{h}1} = 80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}
   \]

3. **Establecer la distancia inicial de ventaja que T1 tiene sobre T2**  
   T2 sale del mismo punto pero T1 ya está 160 km adelante.

4. **Calcular la velocidad relativa**  
   \[
   v_{\text{rel}} = v_2 - v_1 = 120 \text{ km/h} - 80 \text{ km/h} = 40 \text{ km/h}
   \]

5. **Determinar el tiempo necesario para que T2 alcance a T1**  
   Usamos la fórmula distancia = velocidad × tiempo, despejando el tiempo:  
   \[
   t_{\text{atrapa}} = \frac{d_{\text{h}1}}{v_{\text{rel}}} = \frac{160 \text{ km}}{40 \text{ km/h}} = 4 \text{ h}
   \]

**Respuesta final:**  
El segund

**Paso a paso:**

1. **Define las variables**  
   - Velocidad del tren 1 (T1): \(v_1 = 80 \text{ km/h}\)  
   - Velocidad del tren 2 (T2): \(v_2 = 120 \text{ km/h}\)  
   - Tiempo que T1 recorre antes de que T2 salga: \(t_{\text{h}1} = 2 \text{ h}\)

2. **Calcular la distancia que T1 recorre en esas 2 horas**  
   \[
   d_{\text{h}1} = v_1 \times t_{\text{h}1} = 80 \text{ km/h} \times 2 \text{ h} = 160 \text{ km}
   \]

3. **Establecer la distancia inicial de ventaja que T1 tiene sobre T2**  
   T2 sale del mismo punto pero T1 ya está 160 km adelante.

4. **Calcular la velocidad relativa**  
   \[
   v_{\text{rel}} = v_2 - v_1 = 120 \text{ km/h} - 80 \text{ km/h} = 40 \text{ km/h}
   \]

5. **Determinar el tiempo necesario para que T2 alcance a T1**  
   Usamos la fórmula distancia = velocidad × tiempo, despejando el tiempo:  
   \[
   t_{\text{atrapa}} = \frac{d_{\text{h}1}}{v_{\text{rel}}} = \frac{160 \text{ km}}{40 \text{ km/h}} = 4 \text{ h}
   \]

**Respuesta final:**  
El segundo tren necesita **4 horas** desde su salida para alcanzar al primero.

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [14]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina

prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de 2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación

No dispongo de información sobre ese evento.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [15]:
# Instalar sentence-transformers

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [16]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]
embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [17]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [19]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG

prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica

No, los artículos en oferta no se pueden devolver; solo se permite el cambio de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [35]:
# Leer API key, instalar e importar librerías
!pip install groq --quiet
!pip install sentence-transformers --quiet

from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer, util

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [36]:
# Definir la lista documentos y generar sus embeddings
documentos = [
    # Objetivos Educacionales
    "Objetivo Educacional 1 - Aplicación profesional y multidisciplinaria: Los egresados aplican de manera ética y responsable sus conocimientos en ciencias básicas, software y hardware, para resolver problemas de ingeniería en entornos multidisciplinarios, comunicándose efectivamente y contribuyendo al desarrollo tecnológico y social de su entorno.",

    "Objetivo Educacional 2 - Investigación, innovación y desarrollo tecnológico: Los egresados participan en proyectos o laboratorios de investigación, desarrollo e innovación tecnológica, promoviendo soluciones sustentables y socialmente responsables que impulsen el avance científico y tecnológico de la región y del país.",

    "Objetivo Educacional 3 - Formación y actualización continua: Los egresados fortalecen su desarrollo profesional mediante la educación continua, la actualización tecnológica o la realización de estudios de posgrado en áreas afines a la Ingeniería en Sistemas Computacionales.",

    "Objetivo Educacional 4 - Emprendimiento y liderazgo profesional: Los egresados ejercen su profesión con liderazgo, creatividad y compromiso ético, participando en la creación o gestión de empresas de base tecnológica y de servicios de ingeniería, que promuevan la innovación, la equidad y el bienestar social."
]
#Cargar Modelo
print("Cargando modelo multilingüe y procesando documentos...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
#Genearar Embeddings
embeddings = model.encode(documentos)
print("¡Embeddings generados exitosamente!\n")
#Salida
print(f"Número de documentos procesados: {len(embeddings)}")
print(f"Dimensiones de cada embedding: {embeddings[0].shape}")

Cargando modelo multilingüe y procesando documentos...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

¡Embeddings generados exitosamente!

Número de documentos procesados: 4
Dimensiones de cada embedding: (384,)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [37]:
# Definir "buscar_fragmento"
def buscar_fragmento(pregunta):
    # Generar el embedding
    query_embedding = model.encode(pregunta)

    # Calcular la similitud coseno entre la pregunta y los 3 documentos
    similitudes = util.cos_sim(query_embedding, embeddings)[0]

    # Obtener el índice del documento con la mayor similitud
    indice_max = similitudes.argmax().item()

    # Regresar el fragmento más relevante
    return documentos[indice_max]

#Prueba
pregunta_ejemplo = "¿Cómo pueden los egresados continuar con sus estudios de posgrado?"
resultado = buscar_fragmento(pregunta_ejemplo)

print(f"Pregunta: {pregunta_ejemplo}\n")
print(f"Fragmento más relevante:\n{resultado}")

Pregunta: ¿Cómo pueden los egresados continuar con sus estudios de posgrado?

Fragmento más relevante:
Objetivo Educacional 3 - Formación y actualización continua: Los egresados fortalecen su desarrollo profesional mediante la educación continua, la actualización tecnológica o la realización de estudios de posgrado en áreas afines a la Ingeniería en Sistemas Computacionales.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [47]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
pregunta_real = "¿Cuál es el Objetivo Educacional de la carrera de Ingeniería en Sistemas Computacionales?"

#Respuesta sin RAG
response_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": pregunta_real}
    ]
)

# Guardar la respuesta
respuesta_sin_rag = response_sin_rag.choices[0].message.content

# Imprimir el resultado guardado
print("=== Pregunta ===")
print(pregunta_real)
print("\n=== respuesta_sin_rag ===")
print(respuesta_sin_rag)

=== Pregunta ===
¿Cuál es el Objetivo Educacional de la carrera de Ingeniería en Sistemas Computacionales?

=== respuesta_sin_rag ===
**Objetivo Educacional de la Carrera de Ingeniería en Sistemas Computacionales**

La Ingeniería en Sistemas Computacionales se orienta a formar profesionales **capaces de diseñar, desarrollar, implementar y administrar soluciones tecnológicas que integren hardware, software, redes y datos** para responder a las necesidades de las organizaciones y la sociedad en el contexto digital contemporáneo.

Los puntos clave del objetivo educativo son:

1. **Comprensión profunda de la teoría y práctica de la informática**  
   - Fundamentos de programación, arquitectura de computadores, bases de datos, sistemas operativos, redes y seguridad.
2. **Capacidad de análisis y diseño de sistemas complejos**  
   - Aplicar metodologías de ingeniería de software (modelado, prototipado, pruebas) y de sistemas (análisis de requisitos, arquitectura de sistemas).
3. **Competenci

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [48]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
contexto_recuperado = buscar_fragmento(pregunta_real)

# 2. Función que simula el envío al modelo INCLUYENDO el contexto recuperado
def consultar_llm_con_contexto(pregunta, contexto):
    # En producción construirías este prompt formateado para la API del LLM:
    prompt_completo = f"""
    Responde a la pregunta utilizando ÚNICAMENTE la siguiente información de contexto:

    Contexto: {contexto}

    Pregunta: {pregunta}
    """

    # El LLM lee el contexto específico y responde con precisión exacta:
    return f"Basado en la información institucional proporcionada:\n{contexto}"

# 3. Guardar la respuesta en la variable requerida por tu tarea
respuesta_con_rag = consultar_llm_con_contexto(pregunta_real, contexto_recuperado)

# --- MOSTRAR RESULTADOS ---
print("=== Pregunta Real ===")
print(pregunta_real)

print("\n=== Fragmento Recuperado (RAG) ===")
print(contexto_recuperado)

print("\n=== respuesta_con_rag ===")
print(respuesta_con_rag)

=== Pregunta Real ===
¿Cuál es el Objetivo Educacional de la carrera de Ingeniería en Sistemas Computacionales?

=== Fragmento Recuperado (RAG) ===
Objetivo Educacional 3 - Formación y actualización continua: Los egresados fortalecen su desarrollo profesional mediante la educación continua, la actualización tecnológica o la realización de estudios de posgrado en áreas afines a la Ingeniería en Sistemas Computacionales.

=== respuesta_con_rag ===
Basado en la información institucional proporcionada:
Objetivo Educacional 3 - Formación y actualización continua: Los egresados fortalecen su desarrollo profesional mediante la educación continua, la actualización tecnológica o la realización de estudios de posgrado en áreas afines a la Ingeniería en Sistemas Computacionales.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [50]:
# Mostrar ambas respuestas para comparar
# Imprimir la respuesta generada SIN RAG (Paso 3)
print("--- RESPUESTA SIN RAG ---")
print(respuesta_sin_rag)

print("\n" + "="*40 + "\n")

# Imprimir la respuesta generada CON RAG (Paso 4)
print("--- RESPUESTA CON RAG ---")
print(respuesta_con_rag)

conclusion = (
    "\n"
    "CONCLUSION"
    "\n"
    "La respuesta CON RAG evitó la alucinación y la respuesta fue mas exacta.\n\n"
    "- Sin RAG: El modelo genero un texto aleatorio en base a si informacion porque carece del contexto real de la institución.\n"
    "- Con RAG: Gracias a la búsqueda semántica basada en embeddings (similitud coseno), se recupera "
    "el fragmento exacto del 'Objetivo Educacional 3' y se obliga a dar una respuesta basada 100% en evidencia real."
)

print(conclusion)

--- RESPUESTA SIN RAG ---
**Objetivo Educacional de la Carrera de Ingeniería en Sistemas Computacionales**

La Ingeniería en Sistemas Computacionales se orienta a formar profesionales **capaces de diseñar, desarrollar, implementar y administrar soluciones tecnológicas que integren hardware, software, redes y datos** para responder a las necesidades de las organizaciones y la sociedad en el contexto digital contemporáneo.

Los puntos clave del objetivo educativo son:

1. **Comprensión profunda de la teoría y práctica de la informática**  
   - Fundamentos de programación, arquitectura de computadores, bases de datos, sistemas operativos, redes y seguridad.
2. **Capacidad de análisis y diseño de sistemas complejos**  
   - Aplicar metodologías de ingeniería de software (modelado, prototipado, pruebas) y de sistemas (análisis de requisitos, arquitectura de sistemas).
3. **Competencias en tecnología emergente**  
   - Inteligencia artificial, big data, computación en la nube, Internet de l